In [7]:
import json

# new_file_path = "/kaggle/working/processed_tax_rules.json"

# with open(new_file_path, "w") as f:
#     json.dump(loaded_rules, f, indent=4)


# print(f"Saved to {new_file_path}")
file_path = "/kaggle/input/datasets/sudshiv1/sudududu/processed_tax_rules.json"


In [8]:
with open(file_path, "r") as f:
    loaded_rules = json.load(f)

print("Number of rules:", len(loaded_rules))
print("Sample rule:", loaded_rules[0])

Number of rules: 20
Sample rule: {'rule_id': 'R01', 'rule_text': 'Business expenses must be directly related to business activities to be deductible.', 'explanation': 'Expenses that are personal or not clearly linked to business operations should not be fully claimed.', 'category': 'Business vs Personal', 'embedding_text': 'Rule: Business expenses must be directly related to business activities to be deductible. Explanation: Expenses that are personal or not clearly linked to business operations should not be fully claimed.'}


In [9]:
for rule in loaded_rules:
    rule["embedding_text"] = f"Rule: {rule['rule_text']} Explanation: {rule['explanation']}"

for i in range(3):
    print(loaded_rules[i]["embedding_text"])
    print()

Rule: Business expenses must be directly related to business activities to be deductible. Explanation: Expenses that are personal or not clearly linked to business operations should not be fully claimed.

Rule: Mixed-use expenses should be claimed proportionally based on business usage. Explanation: If an expense or asset is used for both personal and business purposes, only the business portion is eligible.

Rule: All business income must be fully and accurately reported. Explanation: Omitting or underreporting income, including cash transactions, can lead to penalties.



# embeddings

In [10]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
embedding_texts = [
    f"Rule: {rule['rule_text']} Explanation: {rule['explanation']}"
    for rule in loaded_rules
]

In [13]:
embeddings = model.encode(embedding_texts, convert_to_numpy=True)

In [14]:
print("Shape of embeddings:", embeddings.shape)

Shape of embeddings: (20, 384)


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity([embeddings[0]], [embeddings[1]])
print("Similarity:", sim[0][0])

Similarity: 0.63018966


In [16]:
for i, rule in enumerate(loaded_rules):
    rule["embedding"] = embeddings[i].tolist()

# embed


In [17]:
!pip install -q faiss-cpu

In [18]:


# Imports
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load dataset
# file_path = "/kaggle/working/processed_tax_rules.json"

with open(file_path, "r") as f:
    loaded_rules = json.load(f)

# Create embedding text
for rule in loaded_rules:
    rule["embedding_text"] = f"Rule: {rule['rule_text']} Explanation: {rule['explanation']}"

embedding_texts = [rule["embedding_text"] for rule in loaded_rules]

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
embeddings = model.encode(embedding_texts, convert_to_numpy=True)
embeddings = embeddings.astype('float32')

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

faiss.normalize_L2(embeddings)
index.add(embeddings)
# Create mapping
id_to_rule = {i: rule for i, rule in enumerate(loaded_rules)}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:

# Retrieval function
def retrieve_rules(query, top_k=4):
    query_embedding = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)
    # query_embedding = model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        rule = id_to_rule[idx]
        results.append({
            "rule_id": rule["rule_id"],
            "rule_text": rule["rule_text"],
            "score": float(distances[0][i])
        })
    return results

# Pretty print
def pretty_print(results):
    print("Retrieved Rules:\n")
    for i, r in enumerate(results, 1):
        print(f"{i}. {r['rule_id']}")
        print(f"   {r['rule_text']}")
        print(f"   Score: {r['score']:.4f}\n")



In [20]:
queries = [
    "A freelancer uses a laptop partly for work and partly for personal use. How should expenses be claimed?",
    "A business claims fuel expenses but has no receipts. Is this valid?",
    "A company delays reporting income to reduce tax liability. Is this acceptable?"
]
# Test query
# query = "I use my laptop for work and personal use"
for query in queries:
    results = retrieve_rules(query)

    pretty_print(results)

Retrieved Rules:

1. R02
   Mixed-use expenses should be claimed proportionally based on business usage.
   Score: 0.5268

2. R01
   Business expenses must be directly related to business activities to be deductible.
   Score: 0.4970

3. R04
   Expenses must be reasonable and justifiable for business purposes.
   Score: 0.4470

4. R05
   Proper documentation is required to support all expense claims.
   Score: 0.4274

Retrieved Rules:

1. R05
   Proper documentation is required to support all expense claims.
   Score: 0.7051

2. R01
   Business expenses must be directly related to business activities to be deductible.
   Score: 0.5882

3. R04
   Expenses must be reasonable and justifiable for business purposes.
   Score: 0.5486

4. R03
   All business income must be fully and accurately reported.
   Score: 0.5387

Retrieved Rules:

1. R08
   Tax returns must be filed within prescribed deadlines.
   Score: 0.5229

2. R03
   All business income must be fully and accurately reported.
   S